# 05 · Sequences, split, evaluation protocol, analysis groups and federated clients

### What this notebook does
1. Orders each user's history by time (ties broken by original file order) and builds sequences.
2. **Chronological leave-one-out split:** last item → test, second-to-last → validation, the rest → training.
3. Fixes the **evaluation protocol** shared by Models 0–4: full-catalog ranking with history masking (main) and a
   fixed set of 100 sampled negatives per user (secondary).
4. Builds the **analysis groups** from the report's Stage 6: sparse / medium / active users and head / mid / tail items
   (popularity counted on **training data only**).
5. Builds a **cold-start variant**: a held-out set of new products removed from training, to test semantic-only
   embeddings.
6. Builds **user-level federated client partitions** with client-size and non-IID statistics (report Stage 1).
7. Runs automated **leakage checks** and saves a report.

### Why
- The report requires "chronological splits" and that "all variants use the same task, split, candidate protocol".
  Fixing all of this now, once, is what makes Models 0–4 comparable.
- Random splits let the model see the future and inflate results; leave-one-out by time mirrors real use.

**Inputs:** `interim/interactions_kcore.parquet`, `processed/items.parquet`
**Outputs (all in `processed/`):** `interactions.parquet`, `sequences.parquet`, `eval/…`, `groups/…`,
`cold_start/…`, `clients/…`, `leakage_report.json`

In [ ]:
# ---------------- Shared setup (identical in every notebook) ----------------
# All notebooks read and write inside one project folder (BASE).
# Colab: BASE lives in your Google Drive so outputs survive runtime restarts.
# Local Jupyter: BASE is ./298A_recsys next to the notebooks.
# Set the environment variable RECSYS_BASE to use any other folder.
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if os.environ.get('RECSYS_BASE'):
    BASE = Path(os.environ['RECSYS_BASE'])
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A_recsys')
else:
    BASE = Path('./298A_recsys').resolve()

RAW, INTERIM, PROCESSED = BASE / 'raw', BASE / 'interim', BASE / 'processed'
REPORTS, FIGS = BASE / 'reports', BASE / 'reports' / 'figures'
# Large downloads go to fast local disk in Colab (not Drive); re-downloaded if the runtime resets.
DOWNLOADS = Path(os.environ.get('RECSYS_DOWNLOAD_DIR', '/content/downloads' if IN_COLAB else str(BASE / 'downloads')))
for d in [RAW, INTERIM, PROCESSED, REPORTS, FIGS, DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot style: one hue per single-series chart, light recessive grid, no chart junk.
PRIMARY, ACCENT, MUTED = '#2a78d6', '#eb6834', '#8a8984'
plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.prop_cycle': matplotlib.cycler(color=[PRIMARY, ACCENT, '#1baf7a', '#eda100']),
})

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight')
    plt.show()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_config():
    cfg_path = BASE / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError('config.json not found - run 01_data_acquisition.ipynb first.')
    return json.loads(cfg_path.read_text())

print('Project folder:', BASE)

In [ ]:
CFG = load_config()
SEED, MAX_LEN = CFG['seed'], CFG['max_seq_len']
inter = pd.read_parquet(INTERIM / 'interactions_kcore.parquet')
items = pd.read_parquet(PROCESSED / 'items.parquet')
index_info = json.loads((INTERIM / 'index_info.json').read_text())
N_ITEMS, N_USERS = index_info['n_items'], index_info['n_users']
for sub in ['eval', 'groups', 'cold_start', 'clients']:
    (PROCESSED / sub).mkdir(exist_ok=True)
print(f'{len(inter):,} interactions | {N_USERS:,} users | {N_ITEMS:,} items')

## 1. Chronological order and leave-one-out split
Amazon timestamps can tie (several reviews written at the same moment). Sorting by `(timestamp, row_id)` breaks
ties by the original file order, so the sequence order is deterministic and identical for every model.

In [ ]:
inter = inter.sort_values(['user_idx', 'timestamp', 'row_id'], kind='mergesort').reset_index(drop=True)
inter['pos'] = inter.groupby('user_idx').cumcount().astype(np.int32)
inter['n_user'] = inter.groupby('user_idx')['item_idx'].transform('size').astype(np.int32)
inter['split'] = np.where(inter.pos == inter.n_user - 1, 'test',
                  np.where(inter.pos == inter.n_user - 2, 'val', 'train'))
print(inter.split.value_counts())
tie_users = inter[inter.duplicated(['user_idx', 'timestamp'], keep=False)].user_idx.nunique()
print(f'Users with at least one timestamp tie (resolved by row_id): {tie_users:,} ({tie_users / N_USERS:.1%})')

## 2. Sequences
One row per user: the full training sequence (with timestamps), the validation target and the test target.
Sequences are stored **untruncated**; the shared data loader (notebook 07) keeps the last `max_seq_len = 50` items.

| Split | Input to the model | Target |
|---|---|---|
| train | prefixes of `train_items` | next item at every position |
| validation | last 50 of `train_items` | `val_item` |
| test | last 50 of `train_items + [val_item]` | `test_item` |

In [ ]:
train = inter[inter.split == 'train']
seq = train.groupby('user_idx').agg(train_items=('item_idx', list), train_ts=('timestamp', list)).reset_index()
val = inter[inter.split == 'val'].set_index('user_idx')
test = inter[inter.split == 'test'].set_index('user_idx')
seq['val_item'] = seq.user_idx.map(val.item_idx).astype(np.int32)
seq['val_ts'] = seq.user_idx.map(val.timestamp).astype(np.int64)
seq['test_item'] = seq.user_idx.map(test.item_idx).astype(np.int32)
seq['test_ts'] = seq.user_idx.map(test.timestamp).astype(np.int64)
seq['n_train'] = seq.train_items.str.len().astype(np.int32)
assert len(seq) == N_USERS, 'every user must have a training sequence'

print('Training sequence length:\n', seq.n_train.describe(percentiles=[.5, .9, .99]).round(1))
print(f'Users whose training history is longer than max_seq_len={MAX_LEN}: {(seq.n_train > MAX_LEN).mean():.2%}')
seq.head()

## 3. Item popularity from training data only
Popularity is used for the head / tail groups and the popularity baseline. Counting validation or test interactions
would leak future information, so only the training split is used.

In [ ]:
train_count = np.bincount(train.item_idx.values, minlength=N_ITEMS + 1).astype(np.int64)
train_count[0] = 0
zero_train_items = int((train_count[1:] == 0).sum())
print(f'Items with zero training interactions (appear only as val/test targets): {zero_train_items:,}')

## 4. Evaluation protocol
**Main protocol: full-catalog ranking.** Score all `N_ITEMS` items, mask items already in the user's input history,
and find the rank of the true next item. Sampled evaluation is known to distort model comparisons
(Krichene & Rendle, 2020), so it is only a secondary check.

**Secondary protocol: 100 sampled negatives.** For each user, 100 items the user never interacted with, drawn
uniformly with a fixed seed. Separate draws for validation and test. Saved to disk so every model uses the same ones.

**Metrics (report table):** Recall@K, NDCG@K, HitRate@K, MRR for K in `eval_ks`; plus coverage and diversity.
With one target per user, Recall@K equals HitRate@K; both are reported for consistency with the report.

In [ ]:
def sample_negatives(seq_df, n_neg, rng):
    out = np.zeros((len(seq_df), n_neg), dtype=np.int32)
    for row, (tr, v, t) in enumerate(zip(seq_df.train_items, seq_df.val_item, seq_df.test_item)):
        seen = set(tr); seen.add(v); seen.add(t)
        picked = []
        while len(picked) < n_neg:
            cand = rng.integers(1, N_ITEMS + 1, size=2 * n_neg)
            for c in cand:
                if c not in seen:
                    seen.add(c); picked.append(c)
                    if len(picked) == n_neg:
                        break
        out[row] = picked
    return out

n_neg = min(CFG['n_sampled_negatives'], N_ITEMS - int(seq.n_train.max()) - 2)
rng = np.random.default_rng(SEED)
neg_val = sample_negatives(seq, n_neg, rng)
neg_test = sample_negatives(seq, n_neg, rng)
np.save(PROCESSED / 'eval' / 'neg_val.npy', neg_val)
np.save(PROCESSED / 'eval' / 'neg_test.npy', neg_test)

eval_protocol = {
    'main': 'full_catalog_ranking',
    'mask_history': True,
    'history_definition': {'val': 'train_items', 'test': 'train_items + [val_item]'},
    'secondary': f'{n_neg}_uniform_sampled_negatives',
    'negatives_files': {'val': 'eval/neg_val.npy', 'test': 'eval/neg_test.npy'},
    'negatives_row_order': 'same as sequences.parquet (sorted by user_idx)',
    'metrics': ['Recall@K', 'NDCG@K', 'HitRate@K', 'MRR', 'Coverage@K', 'ILD@K (category)'],
    'ks': CFG['eval_ks'],
    'max_seq_len': MAX_LEN,
    'padding_idx': 0,
    'model_selection': 'early stopping on validation NDCG@10; test evaluated once at the end',
    'seeds_to_report': [42, 43, 44],
}
save_json(eval_protocol, PROCESSED / 'eval' / 'eval_protocol.json')
print(json.dumps(eval_protocol, indent=2))

## 5. Analysis groups
### 5a. Users by training-history length
Model 1's semantic layer is expected to help most for short histories (HSUGA's motivation), so we fix the groups now.

In [ ]:
edges = CFG['user_length_bins'] + [np.inf]
labels = []
for lo, hi in zip(edges[:-1], edges[1:]):
    labels.append(f'{lo}+' if hi == np.inf else f'{lo}-{int(hi) - 1}')
names = ['short', 'medium', 'long'] if len(labels) == 3 else labels
user_groups = seq[['user_idx', 'n_train']].copy()
user_groups['length_bin'] = pd.cut(user_groups.n_train, bins=edges, right=False, labels=labels).astype(str)
user_groups['user_group'] = user_groups.length_bin.map(dict(zip(labels, names)))
user_groups.loc[user_groups.n_train < edges[0], 'user_group'] = 'short'
user_groups.to_parquet(PROCESSED / 'groups' / 'user_groups.parquet', index=False)
display(user_groups.groupby(['user_group', 'length_bin']).size().rename('users').to_frame())

### 5b. Items by training popularity
Head = top 20% most popular items, tail = bottom 50%, mid = the rest (fractions in `config.json`).

In [ ]:
order = np.lexsort((np.arange(1, N_ITEMS + 1), -train_count[1:]))  # by count desc, then item_idx
ranked_items = np.arange(1, N_ITEMS + 1)[order]
n_head = int(np.ceil(CFG['item_pop']['head_frac'] * N_ITEMS))
n_tail = int(np.floor(CFG['item_pop']['tail_frac'] * N_ITEMS))
grp = np.full(N_ITEMS + 1, 'mid', dtype=object); grp[0] = 'padding'
grp[ranked_items[:n_head]] = 'head'
grp[ranked_items[N_ITEMS - n_tail:]] = 'tail'
item_groups = pd.DataFrame({'item_idx': np.arange(1, N_ITEMS + 1), 'train_count': train_count[1:],
                            'item_group': grp[1:]})
item_groups.to_parquet(PROCESSED / 'groups' / 'item_groups.parquet', index=False)
share = item_groups.groupby('item_group').train_count.sum() / train_count.sum()
display(pd.DataFrame({'items': item_groups.item_group.value_counts(), 'share_of_train_interactions': share.round(3)}))

### 5c. Evaluation cases with group labels
One row per (user, split) evaluation case. Models 0–4 join their per-user ranks to this table to get the breakdowns.

In [ ]:
cases = []
for split, col in [('val', 'val_item'), ('test', 'test_item')]:
    c = seq[['user_idx', col]].rename(columns={col: 'target'})
    c['split'] = split
    cases.append(c)
cases = pd.concat(cases, ignore_index=True)
cases = cases.merge(user_groups[['user_idx', 'user_group']], on='user_idx')
cases['target_item_group'] = grp[cases.target.values]
cases['target_train_count'] = train_count[cases.target.values]
cases.to_parquet(PROCESSED / 'eval' / 'eval_cases.parquet', index=False)
pd.crosstab(cases[cases.split == 'test'].user_group, cases[cases.split == 'test'].target_item_group, margins=True)

## 6. Cold-start variant
**Goal:** test whether semantic embeddings can recommend products the model has *never* seen in training.

**Construction**
1. Pool = the newest 30% of products (by first interaction time): new products are the realistic cold-start case.
2. Cold set `C` = random 5% of all products drawn from that pool (fixed seed).
3. Cold training data = every user's training sequence with `C` items removed; validation targets in `C` removed.
4. Cold test cases = users whose **test item is in `C`**.

This is a separate variant used only for the cold-start experiment. The main split above is unchanged.
Model 0 (ID-only) cannot score `C` items at all; that is exactly the comparison the report asks for.

In [ ]:
first_ts = inter.groupby('item_idx').timestamp.min()
pool_cut = first_ts.quantile(1 - CFG['cold_start']['newest_pool_frac'])
pool = np.sort(first_ts[first_ts >= pool_cut].index.values)
n_cold = min(int(round(CFG['cold_start']['item_frac'] * N_ITEMS)), len(pool))
cold_items = np.sort(np.random.default_rng(SEED).choice(pool, size=n_cold, replace=False)).astype(np.int32)
cold_set = set(cold_items.tolist())

cold = seq[['user_idx', 'train_items', 'val_item', 'test_item']].copy()
cold['train_items'] = cold.train_items.apply(lambda l: [i for i in l if i not in cold_set])
cold['val_item'] = np.where(cold.val_item.isin(cold_set), -1, cold.val_item).astype(np.int32)   # -1 = no val case
cold['test_is_cold'] = cold.test_item.isin(cold_set)
cold['n_train'] = cold.train_items.str.len()
cold['usable_for_training'] = cold.n_train >= 1

np.save(PROCESSED / 'cold_start' / 'cold_items.npy', cold_items)
cold.to_parquet(PROCESSED / 'cold_start' / 'cold_sequences.parquet', index=False)
cold_info = {'n_cold_items': int(n_cold), 'pool_size': int(len(pool)),
             'pool_first_ts_cutoff': str(pd.to_datetime(pool_cut, unit='ms', utc=True)),
             'n_cold_test_cases': int(cold.test_is_cold.sum()),
             'n_users_losing_val_case': int((cold.val_item == -1).sum()),
             'n_users_unusable_for_training': int((~cold.usable_for_training).sum()),
             'train_interactions_removed': int(seq.n_train.sum() - cold.n_train.sum())}
save_json(cold_info, PROCESSED / 'cold_start' / 'cold_start_info.json')
print(json.dumps(cold_info, indent=2))
if cold_info['n_cold_test_cases'] < 500:
    print('WARNING: fewer than 500 cold test cases; consider increasing cold_start.item_frac in config.json.')

## 7. Federated client partitions
The report's Model 2 treats **each user as one client** (no raw history leaves the device). We save one row per
client with its local data size and how different its category mix is from the global mix (non-IID degree,
measured as KL divergence). The report asks to "report client sizes ... and non-IID behavior".

In [ ]:
cat_col = 'cat_l2' if index_info['subset']['kind'] == 'category' else 'cat_l1'
item_cat = items.set_index('item_idx')[cat_col].fillna('unknown')
tr = train[['user_idx', 'item_idx']].assign(cat=lambda d: d.item_idx.map(item_cat))
global_p = tr.cat.value_counts(normalize=True)
uc = tr.groupby(['user_idx', 'cat']).size().rename('cnt').reset_index()
uc['p'] = uc.cnt / uc.groupby('user_idx').cnt.transform('sum')
uc['kl_term'] = uc.p * np.log(uc.p / uc.cat.map(global_p))
kl = uc.groupby('user_idx').kl_term.sum()
dominant = uc.sort_values(['user_idx', 'cnt'], ascending=[True, False]).drop_duplicates('user_idx').set_index('user_idx').cat

clients = seq[['user_idx', 'n_train']].rename(columns={'user_idx': 'client_id', 'n_train': 'n_local_train'})
clients['n_local_total'] = clients.n_local_train + 2
clients['size_group'] = clients.client_id.map(user_groups.set_index('user_idx').user_group)
clients['n_categories'] = clients.client_id.map(uc.groupby('user_idx').size())
clients['dominant_category'] = clients.client_id.map(dominant)
clients['kl_to_global'] = clients.client_id.map(kl).round(4)
clients.to_parquet(PROCESSED / 'clients' / 'clients.parquet', index=False)

def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))

client_stats = {
    'n_clients': int(len(clients)), 'category_level_for_non_iid': cat_col,
    'local_train_size': clients.n_local_train.describe(percentiles=[.1, .5, .9]).round(2).to_dict(),
    'local_size_gini': round(gini(clients.n_local_train), 4),
    'kl_to_global': clients.kl_to_global.describe(percentiles=[.1, .5, .9]).round(4).to_dict(),
    'share_clients_single_category': round(float((clients.n_categories == 1).mean()), 4),
}
save_json(client_stats, PROCESSED / 'clients' / 'client_stats.json')
print(json.dumps(client_stats, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(clients.n_local_train.clip(upper=clients.n_local_train.quantile(0.99)), bins=40, color=PRIMARY)
axes[0].set_title('Client size (local training items)'); axes[0].set_xlabel('items (clipped at p99)')
axes[1].hist(clients.kl_to_global, bins=40, color=PRIMARY)
axes[1].set_title('Non-IID degree: KL(client || global)'); axes[1].set_xlabel(f'KL over {cat_col}')
savefig('05_clients')

## 8. Leakage and integrity checks
Every check must pass. If one fails, the notebook stops, so a broken split can never reach model training.

In [ ]:
checks = {}
def check(name, cond):
    checks[name] = bool(cond)
    print(('PASS ' if cond else 'FAIL ') + name)

per_user_split = inter.groupby(['user_idx', 'split']).size().unstack(fill_value=0)
check('every user has exactly one val and one test interaction',
      (per_user_split['val'] == 1).all() and (per_user_split['test'] == 1).all())
check(f'every user has >= {CFG["kcore"] - 2} training interactions', (per_user_split['train'] >= CFG['kcore'] - 2).all())
max_train_ts = seq.train_ts.apply(max)
check('chronology: max(train_ts) <= val_ts <= test_ts for all users',
      ((max_train_ts <= seq.val_ts) & (seq.val_ts <= seq.test_ts)).all())
check('val item never in that user\'s training sequence',
      not any(v in set(t) for v, t in zip(seq.val_item, seq.train_items)))
check('test item never in that user\'s training sequence or val',
      not any(x in set(t) or x == v for x, v, t in zip(seq.test_item, seq.val_item, seq.train_items)))
check('item ids within 1..N_ITEMS (0 reserved for padding)',
      inter.item_idx.between(1, N_ITEMS).all() and seq.train_items.apply(min).min() >= 1)
check('train + val + test == all interactions', int(seq.n_train.sum()) + 2 * N_USERS == len(inter))
recount = np.bincount(inter.loc[inter.split == 'train', 'item_idx'], minlength=N_ITEMS + 1)
check('item popularity computed from training split only', np.array_equal(recount, train_count))
bad_neg = 0
for row, (tr_, v, t) in enumerate(zip(seq.train_items, seq.val_item, seq.test_item)):
    hist = set(tr_) | {v, t}
    bad_neg += len(hist.intersection(neg_val[row])) + len(hist.intersection(neg_test[row]))
check('sampled negatives never include a history item or a target', bad_neg == 0)
check('cold items absent from cold-variant training sequences',
      not any(i in cold_set for l in cold.train_items for i in l))
check('cold items absent from cold-variant validation targets', not cold.val_item.isin(cold_set).any())
check('item groups use training counts only', np.array_equal(item_groups.train_count.values, train_count[1:]))

save_json({'checks': checks, 'all_passed': all(checks.values())}, PROCESSED / 'leakage_report.json')
assert all(checks.values()), 'Leakage / integrity check failed - fix before training any model.'

## Save interactions and sequences

In [ ]:
inter_out = inter[['user_idx', 'item_idx', 'rating', 'timestamp', 'pos', 'split', 'row_id']]
inter_out.to_parquet(PROCESSED / 'interactions.parquet', index=False)
seq.to_parquet(PROCESSED / 'sequences.parquet', index=False)
print('Saved processed/interactions.parquet and processed/sequences.parquet')

## Summary
| File | Used by |
|---|---|
| `sequences.parquet` | all models (training and evaluation inputs) |
| `eval/eval_protocol.json`, `eval/neg_*.npy`, `eval/eval_cases.parquet` | all models (identical evaluation) |
| `groups/user_groups.parquet`, `groups/item_groups.parquet` | sparse-user and long-tail analysis |
| `cold_start/*` | cold-start experiment (Model 1 semantic-only vs. Model 0) |
| `clients/*` | Model 2 (federated) and Model 4 (DP-FedAvg) |
| `leakage_report.json` | proof for the report that the split is temporally valid |

**Next:** `06_item_text_preparation.ipynb`